# RAW REMMLOT INGESTION AND VALIDATION

* Reads the 95 LocoEventRec XLSX exports and writes `telemetry_raw_v2.pkl`.
* Includes tracker finding:
  * Validity specification: Bounds tightened where the INGEST bounds let physically impossible values through: KWH (a one-second energy increment) capped at 2.0 kWh, battery voltage floored at 70 V (the recorder runs from the battery, so a zero reading while recording is a dropout), DIST_M capped at 60 m per second (consistent with the 200 km/h speed bound).
  The AMP bound is left at 1,500 A pending the Section 7 report.
  * Sentinels scoped: 956 and 975 were treated as sentinels on every channel, which also blanked legitimate DC link voltages and catenary currents. Only the integer-register sentinels (±32767, −32768, 65535) are now global; 956 and 975 are caught by the range bounds on the channels where they are impossible.
  * Simulation flag coerced: `simulation_mode_status` is mapped through the same YES/NO/ON/OFF table as the binary channels, and its raw values are reported, so a text-valued flag can no longer pass the filter silently.
  * Extremes report (`extremes_report.txt`) written before the bounds are applied: counts above the thresholds, the rows themselves, and the gap preceding each implausible KWH value.

In [ ]:
import os, re
from collections import Counter
import numpy as np
import pandas as pd
from openpyxl import load_workbook

CFG = dict(
    src      = "DATA/37011-TEL-ONEMONTH",     # folder holding the 95 XLSX files
    out      = "TELEMETRY/build",
    out_name = "telemetry_raw_v2.pkl",
    keep_simulation = False,
)

In [ ]:
# 1. FORMAT CONSTANTS
# The data column is labelled DD/MM/YYYY but the values are MM/DD/YYYY. The header block genuinely is DD/MM. Never use format='mixed'.
DATA_TS_FORMAT = "%m/%d/%Y %H:%M:%S"
HEADER_TS_FORMAT = "%d/%m/%Y %H:%M:%S"
HEADER_ROW = 3
FIRST_DATA_ROW = 4
TS_COL = "Date/Time (DD/MM/YYYY HH:MM:SS)"

In [ ]:
# 2. CHANNELS
NUMERIC_SRC = [
    "Loco_speed (KMPH)", "Distance_Travelled (Meters)", "OHE_Volt_KV (KV)",
    "OHE_Current (Amps)", "Energy_Consumed (Kwh)", "Battery_Volt (Volts)",
    "BP_Pressure (Kg/cm2)", "TE_BE_DemandPer (%)", "TE_BE_BG1_KN (KN)",
    "TE_BE_BG2_KN (KN)", "Bogie1_DCLV (Volts)", "Bogie2_DCLV (Volts)",
]

BINARY_SRC = [
    "Panto_Status", "VCB_Status", "Motoring", "DB", "Traction_Interlock",
    "Start_Run_Interlock", "Emergency_Brake_Applied", "Isolating_Emg_Exhaust",
    "Brk_Ele_Hlth", "Parking_Break_Pr_Sw", "Air_Flow_Pr_Sw", "BC1_Pr_Sw",
    "BC2_Pr_Sw", "MR_Low_Pr_Sw", "Panto_1_Pr_Sw", "Panto_2_Pr_Sw",
    "Vig_Warning", "Vig_Emg_Brk",
    "Wheel_Slip_TM1", "Wheel_Slip_TM2", "Wheel_Slip_TM3",
    "Wheel_Slip_TM4", "Wheel_Slip_TM5", "Wheel_Slip_TM6",
    "CAB1_Key_in_C", "CAB1_ParkBrake", "CAB1_FwSand", "CAB1_Key_in_D",
    "CAB1_ComprOff", "CAB1_DirFor", "CAB1_DirRev", "CAB1_MaxTELimit",
    "CAB1_EmgStop", "CAB1_FtSwLocBrk",
    "CAB2_Key_in_C", "CAB2_ParkBrake", "CAB2_FwSand", "CAB2_Key_in_D",
    "CAB2_ComprOff", "CAB2_DirFor", "CAB2_DirRev", "CAB2_MaxTELimit",
    "CAB2_EmgStop", "CAB2_FtSwLocBrk",
]

CATEGORICAL = ["OP_Mode (NUM)"]
CONTROL = ["simulation_mode_status"]
RETAIN = [TS_COL] + NUMERIC_SRC + BINARY_SRC + CATEGORICAL + CONTROL

SHORT = {
    TS_COL: "DATETIME",
    "Loco_speed (KMPH)": "KMH", "Distance_Travelled (Meters)": "DIST_M",
    "OHE_Volt_KV (KV)": "KV", "OHE_Current (Amps)": "AMP",
    "Energy_Consumed (Kwh)": "KWH", "Battery_Volt (Volts)": "BV",
    "BP_Pressure (Kg/cm2)": "PRESSURE", "TE_BE_DemandPer (%)": "TRACTION_PCT",
    "TE_BE_BG1_KN (KN)": "BG1_KN", "TE_BE_BG2_KN (KN)": "BG2_KN",
    "Bogie1_DCLV (Volts)": "BG1_DCLV", "Bogie2_DCLV (Volts)": "BG2_DCLV",
    "OP_Mode (NUM)": "OP", "simulation_mode_status": "SIM",
}
NUMERIC = [SHORT[c] for c in NUMERIC_SRC]

BINARY_MAP = {
    "YES": 1, "NO": 0, "ON": 1, "OFF": 0, "APPLIED": 1, "RELEASED": 0,
    "TRUE": 1, "FALSE": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0,
}

In [ ]:
# 3. PHYSICAL VALIDITY SPECIFICATION
VALID_RANGE = {
    "KMH":          (0, 200),     # WAP-7 maximum service speed 140 km/h
    "DIST_M":       (0, 60),      # metres per one-second record; 200 km/h = 55.6 m
    "KV":           (0, 32),      # 25 kV nominal overhead supply
    "AMP":          (0, 1500),    # AUTHOR DECISION after the Section 7 report
    "KWH":          (0, 2.0),     # kWh per one-second record; 2.0 kWh = 7.2 MW
    "BV":           (70, 140),    # 110 V nominal battery; recorder runs from it
    "PRESSURE":     (0, 12),      # kg/cm2 brake pipe
    "TRACTION_PCT": (0, 100),
    "BG1_KN":       (-600, 600),
    "BG2_KN":       (-600, 600),
    "BG1_DCLV":     (0, 4000),    # DC link, about 2,800 V nominal
    "BG2_DCLV":     (0, 4000)
}

# Integer-register sentinels only. 956 and 975 are no longer global: on KMH and DIST_M the range bounds already catch them, and on AMP and the DC link voltages they are legitimate readings.
SENTINELS = {32767, -32767, -32768, 65535}

In [ ]:
# 4. READING
PART_RE = re.compile(r"_(\d{6})PART(\d+)\.XLSX$", re.IGNORECASE)

def parse_filename(path):
    m = PART_RE.search(os.path.basename(path))
    return (m.group(1), int(m.group(2))) if m else ("unknown", 0)

def read_header_range(ws):
    row = next(ws.iter_rows(min_row=2, max_row=2, values_only=True))
    frm = to = None
    for i, cell in enumerate(row):
        if cell is None:
            continue
        label = str(cell).strip().lower()
        if label.startswith("from date") and i + 1 < len(row):
            frm = str(row[i + 1]).strip()
        elif label.startswith("to date") and i + 1 < len(row):
            to = str(row[i + 1]).strip()
    def _p(v):
        try:
            return pd.to_datetime(v, format=HEADER_TS_FORMAT)
        except Exception:
            return pd.NaT
    return _p(frm), _p(to)

def read_one(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb.active
    hdr_from, hdr_to = read_header_range(ws)
    header = next(ws.iter_rows(min_row=HEADER_ROW, max_row=HEADER_ROW, values_only=True))
    index = {name: i for i, name in enumerate(header) if name is not None}
    missing = [c for c in RETAIN if c not in index]
    keep = [c for c in RETAIN if c in index]
    positions = [index[c] for c in keep]
    records = [[row[p] for p in positions]
               for row in ws.iter_rows(min_row=FIRST_DATA_ROW, values_only=True)
               if row[0] is not None]
    wb.close()
    df = pd.DataFrame(records, columns=keep)
    meta = dict(file=os.path.basename(path), n_columns_in_file=len(index), n_rows=len(df), missing_channels=missing,hdr_from=hdr_from, hdr_to=hdr_to)
    return df, meta

In [ ]:
# 5. CLEANING
def to_binary(s):
    return s.astype(str).str.strip().str.upper().map(BINARY_MAP).astype("Int8")

def coerce(df, meta):
    df = df.rename(columns=SHORT)
    df["DATETIME"] = pd.to_datetime(df["DATETIME"], format=DATA_TS_FORMAT, errors="raise")
    for c in NUMERIC:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in BINARY_SRC:
        if c in df.columns:
            df[c] = to_binary(df[c])
    if "SIM" in df.columns:
        meta["sim_raw"] = Counter(df["SIM"].astype(str).str.strip())
        df["SIM"] = to_binary(df["SIM"])
    if "OP" in df.columns:
        df["OP"] = df["OP"].astype(str).str.strip().str.upper()
    return df

def apply_validity(df):
    """Blank implausible values; return the frame and a long anomaly log."""
    log = []
    for col, (lo, hi) in VALID_RANGE.items():
        if col not in df.columns:
            continue
        sent = df[col].isin(SENTINELS)
        rng = ((df[col] < lo) | (df[col] > hi)) & ~sent
        for reason, bad in (("sentinel", sent), ("out of range", rng)):
            if bad.any():
                log.append(pd.DataFrame({
                    "DATETIME": df.loc[bad, "DATETIME"].values, "channel": col,
                    "value": df.loc[bad, col].values, "reason": reason,
                    "low": lo, "high": hi}))
        df.loc[sent | rng, col] = np.nan
    return df, (pd.concat(log, ignore_index=True) if log else pd.DataFrame())

def derive(df):
    """ACCELERATION_MS2 in m/s^2 (N-7). POWER_KW deliberately not created (N-8)."""
    dt = df["DATETIME"].diff().dt.total_seconds()
    acc = (df["KMH"].diff() / 3.6) / dt
    acc[dt != 1] = np.nan
    df["ACCELERATION_MS2"] = acc
    return df

In [ ]:
# 6. EXTREMES REPORT
def extremes_report(df):
    L = []; w = L.append
    gap = df["DATETIME"].diff().dt.total_seconds()
    june = df["DATETIME"].between("2025-06-01", "2025-06-30 23:59:59")
    ctx = ["DATETIME", "OP", "KWH", "AMP", "KV", "KMH", "BV", "VCB_Status"]
    ctx = [c for c in ctx if c in df.columns]

    w("EXTREMES BEFORE VALIDITY BOUNDS")
    w("=" * 72)
    for col, op, thr in [("KWH", ">", 1.5), ("KWH", ">", 2.0), ("KWH", ">", 5.0),
                         ("AMP", ">", 300), ("AMP", ">", 500), ("AMP", ">", 1000),
                         ("BV", "<", 70), ("BV", "==", 0),
                         ("DIST_M", ">", 60)]:
        s = df[col]
        m = s > thr if op == ">" else s < thr if op == "<" else s == thr
        w(f"  {col:8s} {op} {thr:<6}: {int(m.sum()):>7,} rows  "
          f"({int((m & june).sum()):,} in June)")
    w("")

    m = df["KWH"] > 2.0
    w(f"KWH > 2.0 kWh per record, with the gap preceding each row ({int(m.sum())} rows)")
    w("-" * 72)
    if m.any():
        t = df.loc[m, ctx].assign(gap_before_s=gap[m])
        w(t.head(40).to_string(index=False))
        w(f"  preceded by a gap > 1 s: {int((gap[m] > 1).sum())} of {int(m.sum())}")
    w("")

    w("Largest AMP values")
    w("-" * 72)
    w(df.nlargest(20, "AMP")[ctx].to_string(index=False))
    w("")

    m = df["BV"] < 70
    w(f"BV below 70 V ({int(m.sum())} rows)")
    w("-" * 72)
    if m.any():
        w(df.loc[m, ctx].head(20).to_string(index=False))
    w("")

    w("Values equal to 956 or 975, by channel (formerly treated as sentinels)")
    w("-" * 72)
    for c in NUMERIC:
        n956, n975 = int((df[c] == 956).sum()), int((df[c] == 975).sum())
        if n956 or n975:
            lo, hi = VALID_RANGE[c]
            w(f"  {c:10s} 956: {n956:>5}   975: {n975:>5}   "
              f"(range {lo} to {hi}: {'caught' if 956 > hi else 'now kept'})")
    return "\n".join(L)

In [ ]:
# 7. MAIN: RUN FUNCTION
def main(cfg):
    src, out = cfg["src"], cfg["out"]
    os.makedirs(out, exist_ok=True)
    paths = sorted((os.path.join(src, f) for f in os.listdir(src)
                    if f.upper().endswith(".XLSX") and "LOCOEVENTREC" in f.upper()),
                   key=parse_filename)
    if not paths:
        raise FileNotFoundError(f"No LocoEventRec XLSX files found in {src}")
    print(f"Found {len(paths)} files")

    frames, metas = [], []
    for i, p in enumerate(paths, 1):
        df, meta = read_one(p)
        df = coerce(df, meta)
        meta["first"], meta["last"] = df["DATETIME"].min(), df["DATETIME"].max()
        frames.append(df); metas.append(meta)
        print(f"  [{i}/{len(paths)}] {meta['file']}  {meta['n_rows']:>6} rows  "
              f"{meta['first']} -> {meta['last']}")
    all_df = pd.concat(frames, ignore_index=True)

    before = len(all_df)
    dup = all_df.duplicated(subset=["DATETIME"], keep="first")
    n_dup = int(dup.sum())
    conflicting = 0
    if n_dup:
        g = all_df[all_df["DATETIME"].isin(all_df.loc[dup, "DATETIME"])]
        payload = [c for c in all_df.columns if c != "DATETIME"]
        conflicting = int((g.groupby("DATETIME")[payload].nunique(dropna=False) > 1)
                          .any(axis=1).sum())
    all_df = all_df[~dup].copy()

    sim_raw = sum((m.get("sim_raw", Counter()) for m in metas), Counter())
    unmapped = all_df["SIM"].isna().sum() if "SIM" in all_df.columns else 0
    n_sim = int((all_df["SIM"] == 1).sum()) if "SIM" in all_df.columns else 0
    if "SIM" in all_df.columns:
        if not cfg["keep_simulation"]:
            all_df = all_df[~(all_df["SIM"] == 1)].copy()
        all_df = all_df.drop(columns=["SIM"])

    all_df = all_df.sort_values("DATETIME").reset_index(drop=True)
    ext = extremes_report(all_df)
    print("\n" + ext)
    with open(os.path.join(out, "extremes_report.txt"), "w") as f:
        f.write(ext + "\n")

    all_df, anomalies = apply_validity(all_df)
    all_df = derive(all_df)

    span = (all_df["DATETIME"].max() - all_df["DATETIME"].min()).total_seconds() + 1
    gaps = all_df["DATETIME"].diff().dt.total_seconds()
    big = gaps[gaps > 1]
    L = []; w = L.append
    w("RAW REMMLOT INGESTION -- COVERAGE REPORT (INGEST2)")
    w("=" * 72)
    w(f"Files read              : {len(paths)}")
    w(f"Rows before de-dup      : {before:,}")
    w(f"Duplicate timestamps    : {n_dup:,}  (conflicting payloads: {conflicting:,})")
    w(f"Simulation flag, raw    : {dict(sim_raw)}")
    w(f"Simulation flag unmapped: {int(unmapped):,}")
    w(f"Simulation-mode rows    : {n_sim:,} "
      f"({'retained' if cfg['keep_simulation'] else 'removed'})")
    w(f"Rows retained           : {len(all_df):,}")
    w(f"Coverage                : {all_df['DATETIME'].min()} -> {all_df['DATETIME'].max()}")
    w(f"Observed seconds        : {len(all_df):,} ({len(all_df) / span * 100:.1f}% of span)")
    w(f"Gaps longer than 1 s    : {len(big):,}")
    w(f"Seconds absent          : {int((gaps.dropna() - 1).sum()):,}")
    w("")
    w("Ten largest gaps:")
    for i, g in big.nlargest(10).items():
        w(f"    after {all_df.loc[i - 1, 'DATETIME']}   {int(g):>8,} s ({g / 3600:.1f} h)")
    w("")
    w("DECLARED VERSUS ACTUAL COVERAGE PER BATCH")
    w("-" * 72)
    bm = {}
    for m in metas:
        b = parse_filename(m["file"])[0]
        d = bm.setdefault(b, dict(hdr_from=m["hdr_from"], hdr_to=m["hdr_to"],
                                  first=m["first"], last=m["last"], parts=0, rows=0))
        d["first"], d["last"] = min(d["first"], m["first"]), max(d["last"], m["last"])
        d["parts"] += 1; d["rows"] += m["n_rows"]
    for b, d in sorted(bm.items()):
        w(f"  batch {b}: {d['parts']} parts, {d['rows']:,} rows")
        w(f"    declared : {d['hdr_from']} -> {d['hdr_to']}")
        w(f"    actual   : {d['first']} -> {d['last']}")
        if pd.notna(d["hdr_from"]) and d["first"] > d["hdr_from"]:
            w(f"    SHORTFALL at start: {(d['first'] - d['hdr_from']).total_seconds() / 3600:.1f} h")
        if pd.notna(d["hdr_to"]) and d["last"] < d["hdr_to"]:
            w(f"    SHORTFALL at end  : {(d['hdr_to'] - d['last']).total_seconds() / 3600:.1f} h")
    w("")
    miss = Counter(c for m in metas for c in m["missing_channels"])
    w("CHANNELS MISSING FROM SOME FILES")
    w("-" * 72)
    w("\n".join(f"  {c}: absent from {n} file(s)" for c, n in miss.most_common())
      if miss else "  none -- all retained channels present in every file")
    w("")
    w("VALUES BLANKED (rows kept)")
    w("-" * 72)
    if len(anomalies):
        t = anomalies.groupby(["channel", "reason"]).size().unstack(fill_value=0)
        t["June"] = anomalies[anomalies.DATETIME.between(
            "2025-06-01", "2025-06-30 23:59:59")].groupby("channel").size()
        w(t.fillna(0).astype(int).to_string())
        w(f"  total: {len(anomalies):,}")
    else:
        w("  none")
    report = "\n".join(L)
    print("\n" + report)
    with open(os.path.join(out, "coverage_report_v2.txt"), "w") as f:
        f.write(report + "\n")
    if len(anomalies):
        anomalies.to_csv(os.path.join(out, "anomaly_report_v2.csv"), index=False)

    path = os.path.join(out, cfg["out_name"])
    all_df.to_pickle(path)
    print(f"\nWritten: {path}")
    return all_df

In [ ]:
# 8. RUN
df = main(CFG)

In [ ]:
# CHECKS
june = df[df.DATETIME.between("2025-06-01", "2025-06-30 23:59:59")]
print(f"June rows: {len(june):,}  (expected 2,149,773)")
print(june[NUMERIC].describe(percentiles=[0.01, 0.99]).T.round(2).to_string())

Final channel selection: the 12 numeric channels and Wheel_Slip_TM2 to TM6. Protection and operating-state channels enter only the ablation.

___
**END**